# Weights & Biases

A refresher on **Weights & Biases (`wandb`)** — the experiment-tracking and MLOps
platform. You add a few lines to a training script and every run's hyperparameters,
metrics, artifacts, and system stats stream to a live dashboard where you can
compare, group, and share them. Think of it as a **flight recorder + lab notebook**
for machine learning.

**Domain:** AI/ML Tooling  ·  **recommended addition**  ·  **runnable:** yes  ·  _online mode needs a free API key; this notebook runs fully offline_

## 1. What & Why

**What it is.** A hosted (with self-host and offline options) service plus a thin
Python library for **tracking ML experiments**. The core loop is tiny: call
`wandb.init()` to start a *run*, then `run.log({...})` your metrics as training
proceeds. Everything is captured automatically — your config (hyperparameters),
metric curves, the git commit, the exact command, pip environment, GPU/CPU
utilization — and rendered as interactive charts you can filter and compare.

Beyond plain logging it bundles the pieces a real ML workflow needs:
**Artifacts** (versioned datasets/models with lineage), **Sweeps** (built-in
hyperparameter search), **Tables** (rich, queryable logged data — images, audio,
predictions), the **Model Registry**, and **Reports** (shareable write-ups that
embed live charts).

**The problem it solves.** ML is empirical: you run *hundreds* of variants and the
question is always "which config gave the best result, and can I reproduce it?"
`print()` statements, scattered TensorBoard dirs, and a spreadsheet of numbers
collapse fast. W&B gives every run a permanent, comparable, shareable record so
"what did we try and what worked" stops living in someone's head.

**Reach for it when** you're training models you'll iterate on, comparing many
runs/hyperparameters, collaborating on a team, running long jobs you want to watch
remotely, or needing reproducibility and model/dataset versioning.

**Don't reach for it when** it's a one-off throwaway script, you're in a strict
air-gapped environment with no sync path (though *offline mode* often still fits),
or you specifically want a self-hosted, framework-native, dependency-light tracker —
where **MLflow** may suit you better.

## 2. Mental Model

**A flight recorder bolted onto your training loop.** Your code keeps running
exactly as before; `wandb` taps the wire and streams a timestamped record of
everything to a dashboard. You don't restructure your training — you *instrument*
it.

```
   YOUR TRAINING SCRIPT                     WEIGHTS & BIASES
 ┌────────────────────────┐              ┌──────────────────────────┐
 │ run = wandb.init(       │  config ──▶ │  Project "my-model"      │
 │     project=..., config)│             │   ├─ Run: lr=0.1  ████▁▁  │
 │ for step in ...:        │  metrics ─▶ │   ├─ Run: lr=0.3  ██▁▁▁▁  │  live
 │   loss = train()        │  (streamed) │   └─ Run: lr=0.03 ███▆▂▁  │  charts
 │   run.log({"loss":loss})│             │  Artifacts (model v3, ...) │
 │ run.finish()            │  files ───▶ │  Sweeps · Tables · Reports │
 └────────────────────────┘              └──────────────────────────┘
```

The hierarchy that matters: **Entity** (you or your team) ▸ **Project** (a body of
work) ▸ **Run** (one execution) ▸ the **logged data** (metrics at each step,
config, summary, artifacts). A run has a *history* (the time series you `log`) and
a *summary* (one headline value per key — usually the best/last). Online, data
streams live; **offline**, it's written to disk and you `wandb sync` it later — same
code either way.

## 3. Key Concepts

- **Run** — one tracked execution (`wandb.init()` → `run.finish()`). Has a unique id,
  a `config`, a streamed `history`, a `summary`, and any logged artifacts/files.
- **Project / Entity** — a Run lives in a *project* under an *entity* (your username
  or team). `entity/project` is the address; all runs in a project are compared
  side by side.
- **`config`** — the run's input knobs (hyperparameters). Set once at `init`; shows up
  as filterable/groupable columns. Treat it as immutable after training starts.
- **`log()` & step** — `run.log({"loss": x})` appends a row to the history. An
  optional `step=` orders points on the x-axis; steps should be **monotonically
  increasing**.
- **`summary`** — one value per key (e.g. `run.summary["best_acc"]`), the number the
  run table sorts on. `log` is the curve; summary is the headline.
- **Artifact** — a *versioned* file or directory (dataset, model weights, eval set)
  with automatic `v0, v1, …` versioning and lineage between runs.
- **Sweep** — built-in hyperparameter search: a YAML/dict defines the search space
  and method (grid / random / **bayes**); `wandb agent` launches workers that each
  run your script with sampled configs.
- **Table** — rich tabular data you log and explore in the UI (predictions, images,
  text, audio), groupable and chartable.
- **`wandb.watch(model)`** — (PyTorch) auto-logs gradients and parameter histograms.
- **Mode** — `online` (default, streams live), `offline` (writes to `./wandb/`, sync
  later), or `disabled` (no-op, handy for tests).

## 4. Setup

Install the library; that's the whole dependency. To use the hosted dashboard you
authenticate once — `wandb login` (or set the `WANDB_API_KEY` env var) using the
free key from your account settings.

This notebook runs in **offline mode** so it needs *no* key and *no* network: runs
are written to a local directory and could be pushed later with `wandb sync`. The
code is identical to online use — only the `WANDB_MODE` env var changes.

In [1]:
# On a fresh environment, uncomment to install:
# %pip install -q wandb scikit-learn

import os, tempfile

# Run fully offline: no API key, no account, no network. Everything is written
# locally and could later be pushed with `wandb sync`. For real use you'd run
# `wandb login` once and drop these two lines (online is the default).
os.environ["WANDB_MODE"] = "offline"
os.environ["WANDB_SILENT"] = "true"                  # quiet the banner; our prints stand out
os.environ["WANDB_DIR"] = tempfile.mkdtemp(prefix="wandb-refresher-")  # keep the repo clean

import wandb

print("wandb  ", wandb.__version__)
print("mode   ", os.environ["WANDB_MODE"])
print("logdir ", os.environ["WANDB_DIR"])

wandb   0.28.0
mode    offline
logdir  /var/folders/p8/sm5jmh055md_zzhhn1mfgyw80000gn/T/wandb-refresher-trnb8csc


## 5. Worked Examples

### Example 1 — Track a training loop

The canonical pattern: open a run with a `config`, `log` metrics each step, stash a
headline number in `summary`, and `finish`. In a notebook **always call
`run.finish()`** — without it the run stays open and the next `init` complains.

In [2]:
import random

random.seed(0)

# A Run is one tracked execution. `config` holds the hyperparameters — they become
# filterable/groupable columns in the dashboard.
run = wandb.init(
    project="wandb-refresher",
    name="demo-sgd",
    config={"lr": 0.1, "epochs": 15, "optimizer": "sgd"},
)
cfg = run.config                      # namespace-like access: cfg.lr, cfg.epochs

loss = 2.0
for epoch in range(cfg.epochs):
    loss = loss * 0.8 + random.uniform(-0.02, 0.02)   # a fake but plausible curve
    acc = 1 - loss / 2
    run.log({"loss": loss, "accuracy": acc}, step=epoch)   # one history row per step

run.summary["best_acc"] = acc         # summary = the single headline number per run
run.finish()                          # flush + close (essential in notebooks)

print(f"logged {cfg.epochs} steps  ·  final loss={loss:.3f}  ·  acc={acc:.3f}")

logged 15 steps  ·  final loss=0.085  ·  acc=0.957


### Example 2 — Log a real model, a Table, and a versioned Artifact

A complete mini-experiment: train a scikit-learn classifier, log its test accuracy,
log a **Table** of sample predictions (sortable/plottable in the UI), and save the
fitted model as a **versioned Artifact** with lineage back to this run.

In [3]:
import joblib, pathlib
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

X, y = load_iris(return_X_y=True)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0)

run = wandb.init(project="wandb-refresher", name="iris-logreg",
                 config={"C": 1.0, "model": "logreg"})

clf = LogisticRegression(max_iter=500, C=run.config.C).fit(Xtr, ytr)
acc = clf.score(Xte, yte)
run.log({"test_accuracy": acc})

# A Table is queryable logged data you explore in the UI (predictions, images, ...).
preds = clf.predict(Xte[:5])
table = wandb.Table(columns=["true", "pred"],
                    data=[[int(t), int(p)] for t, p in zip(yte[:5], preds)])
run.log({"predictions": table})

# An Artifact is a versioned file/dir (datasets, model weights) with lineage.
path = pathlib.Path(os.environ["WANDB_DIR"]) / "iris_logreg.joblib"
joblib.dump(clf, path)
art = wandb.Artifact("iris-model", type="model", metadata={"acc": acc})
art.add_file(str(path))
run.log_artifact(art)

run.finish()
print(f"test accuracy = {acc:.3f}  ·  logged 1 table + 1 model artifact (iris-model:v0)")

test accuracy = 0.978  ·  logged 1 table + 1 model artifact (iris-model:v0)


### Example 3 — Sweeps (hyperparameter search) and going online

**Sweeps** are W&B's built-in hyperparameter search. You declare a search space and
a method (`grid`, `random`, or `bayes`), register it, and launch agents that each
call your training function with a sampled `config`. The shape (this needs a logged-in
account to actually run, so it's shown, not executed):

```python
sweep_config = {
    "method": "bayes",                                   # grid | random | bayes
    "metric": {"name": "test_accuracy", "goal": "maximize"},
    "parameters": {
        "C":   {"distribution": "log_uniform_values", "min": 1e-3, "max": 1e2},
        "model": {"value": "logreg"},
    },
}

def train():
    run = wandb.init()            # the agent injects the sampled config here
    clf = LogisticRegression(C=run.config.C, max_iter=500).fit(Xtr, ytr)
    run.log({"test_accuracy": clf.score(Xte, yte)})

sweep_id = wandb.sweep(sweep_config, project="wandb-refresher")
wandb.agent(sweep_id, function=train, count=20)          # run 20 trials
```

The cell below runs the same logging code **online** — but only if a key is present,
so the notebook still executes end-to-end without an account.

In [4]:
# Identical code streams to wandb.ai the moment an API key exists. Gated on the env
# var so the notebook runs top-to-bottom either way.
if os.getenv("WANDB_API_KEY"):
    online = wandb.init(project="wandb-refresher", name="online-demo", mode="online")
    online.log({"hello": 1.0})
    online.finish()
    print("logged an online run — open the wandb.ai URL printed above.")
else:
    print("WANDB_API_KEY not set — skipping the online run.")
    print("The offline examples above already executed and wrote to:",
          os.environ["WANDB_DIR"])

WANDB_API_KEY not set — skipping the online run.
The offline examples above already executed and wrote to: /var/folders/p8/sm5jmh055md_zzhhn1mfgyw80000gn/T/wandb-refresher-trnb8csc


## 6. Gotchas & Pitfalls

- **Forgetting `run.finish()` in notebooks.** A cell that re-runs `init` without
  finishing the previous run leaves it open and warns/misbehaves. Always finish (or
  use `with wandb.init() as run:`).
- **Non-monotonic `step`.** History `step` must increase. Logging two different
  metrics that each call `log` with their own counter can clobber the x-axis — log
  related metrics in one `log({...})` call, or let W&B auto-increment.
- **Logging every iteration.** Logging high-frequency scalars every minibatch floods
  the backend and slows training. Log every N steps; aggregate the rest.
- **`config` is for inputs, not outputs.** Put hyperparameters in `config` and
  results in `log`/`summary`. Don't mutate `config` mid-run — it represents the run's
  fixed inputs.
- **Online mode can block.** With a flaky network, `init`/`log` may stall. Use
  `WANDB_MODE=offline` for unreliable connections and `wandb sync` afterward, or
  `mode="disabled"` in unit tests so W&B becomes a no-op.
- **Entity vs project confusion.** Runs land under your *personal* entity by default;
  for a team you must pass `entity="team-name"` or they won't appear where colleagues
  look.
- **Huge artifacts.** Artifacts are versioned and stored; logging multi-GB checkpoints
  every epoch balloons storage. Version deliberately and use `aliases` (`latest`,
  `best`) instead of keeping everything.
- **Secrets in `config`/logs.** `config` and logged values are uploaded — never put
  API keys or PII there.
- **Offline runs aren't in the cloud yet.** Offline writes to `./wandb/`; nothing
  appears online until you run `wandb sync <run-dir>`.

## 7. When to Use vs Alternatives

| Option | Best for | Trade-off vs W&B |
|---|---|---|
| **Weights & Biases** | Polished hosted tracking, sweeps, artifacts, collaboration, reports | SaaS-first (self-host is enterprise); another account/dependency; data leaves your machine unless offline/self-hosted |
| **MLflow** | Open-source, self-hosted tracking + model registry; framework-agnostic | You run/maintain the server; UI and collaboration less slick; no built-in sweeps |
| **TensorBoard** | Lightweight local metric/graph viz, deep TF/PyTorch integration | No experiment *management*, run comparison at scale, sweeps, or sharing; local-first |
| **Neptune / Comet / ClearML** | Direct W&B-style competitors; ClearML adds orchestration/pipelines | Similar SaaS trade-offs; smaller ecosystems; pick on pricing/feature fit |
| **Plain CSV / spreadsheet** | A handful of runs, zero dependencies | Falls apart past a few experiments; no charts, search, versioning, or collaboration |

**Rule of thumb:** if you're iterating on models with a team and want sweeps,
artifacts, and shareable dashboards out of the box, reach for **W&B**. If you need
fully self-hosted/open-source and will run the infra, **MLflow**. For quick local
curves inside one framework, **TensorBoard** is enough.

## 8. Resources

- **Official docs (Developer Guide)** — concepts, integrations, full API: https://docs.wandb.ai/
- **Quickstart** — instrument a script in five minutes: https://docs.wandb.ai/quickstart/
- **Sweeps guide** — search spaces, methods, agents: https://docs.wandb.ai/guides/sweeps/
- **Artifacts guide** — dataset/model versioning and lineage: https://docs.wandb.ai/guides/artifacts/
- **`wandb` GitHub repo** — source, issues, examples: https://github.com/wandb/wandb